In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Tugas5-BigData") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [8]:
import pandas as pd


df_transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv", header=True, inferSchema=True)
df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan":[45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

print("Data transaksi awal:")
df_transaksi.show(5)
print("Data target cabang:")
df_target.show()

Data transaksi awal:
+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows

Data target cabang:
+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      5

**A. Join & Perbandingan Target**

In [9]:
df_pendapatan_kota = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)
df_Perbandingan_Target = df_pendapatan_kota.join(df_target, on="kota", how="inner")


df_Perbandingan_Target = df_Perbandingan_Target.withColumn(
    "pencapaian_persen",
    (col("total_pendapatan") / col("target_bulanan") * 100)
)
df_Perbandingan_Target.orderBy(col("pencapaian_persen").desc()).show()

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



**B. Window Function — Kategori Terlaris per Kota**

In [10]:
df_kota_kategori = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum("pendapatan").alias("pendapatan_kategori")
)

window_spec = Window.partitionBy("kota").orderBy(col("pendapatan_kategori").desc())
df_ranked = df_kota_kategori.withColumn("peringkat", row_number().over(window_spec))
df_Kategori_Terlaris_Perkota = df_ranked.filter(col("peringkat") == 1).drop("peringkat")
df_Kategori_Terlaris_Perkota.orderBy("kota").show()

+----------+--------------------+-------------------+
|      kota|            kategori|pendapatan_kategori|
+----------+--------------------+-------------------+
|  Magelang|Kesehatan & Kecan...|            7275000|
| Purworejo|Kesehatan & Kecan...|           10075000|
|  Semarang|        Rumah Tangga|           11125000|
|      Solo|Kesehatan & Kecan...|            8425000|
|Yogyakarta|             Fashion|           13325000|
+----------+--------------------+-------------------+



**C. Spark SQL**

In [11]:
df_transaksi.createOrReplaceTempView("view_transaksi")
df_target.createOrReplaceTempView("view_target")

df_hasil_c = spark.sql("""
    SELECT t.kota, tg.pic_cabang, COUNT(t.order_id) AS jumlah_transaksi
    FROM view_transaksi t
    JOIN view_target tg ON t.kota = tg.kota
    GROUP BY t.kota, tg.pic_cabang
    ORDER BY jumlah_transaksi DESC
""")

df_hasil_c.show()

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



**D. Kesimpulan**

Dari hasil analisis data pada bagian A dan B, cabang Purworejo di bawah pimpinan PIC Fitri sukses menempati posisi puncak sebagai cabang berkinerja terbaik. Cabang ini mengumpulkan total pendapatan sebesar Rp45.650.000 dan menjadi satu-satunya wilayah yang berhasil melampaui target bulanan Rp30.000.000 secara signifikan dengan persentase pencapaian menyentuh angka 152,16%. Keberhasilan luar biasa ini disokong penuh oleh tingginya penjualan pada kategori produk Kesehatan & Kecantikan yang menyumbangkan omset sebesar Rp10.075.000 di kota tersebut. Sementara itu, untuk wilayah Magelang di bawah pimpinan PIC Rani, performanya berada di peringkat keempat dengan total pendapatan Rp31.650.000 (pencapaian 70,33%), di mana motor penggerak utamanya juga didominasi oleh kategori Kesehatan & Kecantikan yang menyumbang angka Rp7.275.000.

Sebaliknya, cabang Kota Semarang yang dipimpin oleh PIC Sari menjadi wilayah yang paling memerlukan perhatian dan evaluasi strategi mendalam dari pihak manajemen. Semarang berada di urutan terbawah dalam efektivitas pemenuhan target bulanan karena hanya mampu mengamankan persentase pencapaian sebesar 69,40%, atau mengumpulkan omset Rp38.175.000 dari target awal yang cukup besar yaitu Rp55.000.000. Meskipun kategori produk Rumah Tangga di Semarang telah memberikan kontribusi performa tunggal yang sangat besar senilai Rp11.125.000, manajemen tetap perlu mengevaluasi faktor penghambat pada kategori produk lainnya. Pihak manajemen disarankan untuk melakukan penyesuaian target berkala agar lebih rasional bagi Semarang atau mereplikasi strategi pemasaran produk kecantikan dari Purworejo untuk diterapkan di Magelang dan Semarang guna mengatrol omset pada periode berikutnya.


**Explorasi**

In [12]:
rata_harga_global = df_transaksi.agg({"harga_satuan": "avg"}).collect()[0][0]
df_premium = df_transaksi.filter(col("harga_satuan") > rata_harga_global)
df_premium_per_kota = df_premium.groupBy("kota").agg(count("order_id").alias("jumlah_transaksi_premium"))
df_eksplorasi_1 = df_premium_per_kota.join(df_target, on="kota", how="inner")

print(f"=== EKSPLORASI 1: PENYERAPAN PRODUK PREMIUM (DI ATAS Rp {rata_harga_global:,.2f}) PER CABANG ===")
df_eksplorasi_1.orderBy(col("jumlah_transaksi_premium").desc()).show()

=== EKSPLORASI 1: PENYERAPAN PRODUK PREMIUM (DI ATAS Rp 80,900.00) PER CABANG ===
+----------+------------------------+--------------+----------+
|      kota|jumlah_transaksi_premium|target_bulanan|pic_cabang|
+----------+------------------------+--------------+----------+
|Yogyakarta|                      51|      60000000|      Joko|
|      Solo|                      42|      40000000|      Bayu|
| Purworejo|                      41|      30000000|     Fitri|
|  Semarang|                      40|      55000000|      Sari|
|  Magelang|                      33|      45000000|      Rani|
+----------+------------------------+--------------+----------+



In [13]:
from pyspark.sql.functions import lead

window_kota = Window.partitionBy("kota").orderBy(col("pendapatan").desc())
df_gap = df_transaksi.withColumn("peringkat", row_number().over(window_kota)) \
                       .withColumn("pendapatan_peringkat_bawahnya", lead("pendapatan", 1).over(window_kota))

df_analisis_gap = df_gap.filter(col("peringkat") == 1) \
                        .withColumn("selisih_gap_ke_peringkat_2", col("pendapatan") - col("pendapatan_peringkat_bawahnya"))

print("=== EKSPLORASI 2: ANALISIS SELISIH (GAP) PENDAPATAN PERINGKAT 1 VS PERINGKAT 2 ===")
df_analisis_gap.select("kota", "kategori", "pendapatan", "pendapatan_peringkat_bawahnya", "selisih_gap_ke_peringkat_2").show()

=== EKSPLORASI 2: ANALISIS SELISIH (GAP) PENDAPATAN PERINGKAT 1 VS PERINGKAT 2 ===
+----------+--------------------+----------+-----------------------------+--------------------------+
|      kota|            kategori|pendapatan|pendapatan_peringkat_bawahnya|selisih_gap_ke_peringkat_2|
+----------+--------------------+----------+-----------------------------+--------------------------+
|  Magelang|Kesehatan & Kecan...|   1350000|                      1200000|                    150000|
| Purworejo|          Elektronik|   1350000|                      1200000|                    150000|
|  Semarang|   Makanan & Minuman|   1350000|                      1350000|                         0|
|      Solo|             Fashion|   1350000|                      1200000|                    150000|
|Yogyakarta|Kesehatan & Kecan...|   1350000|                      1350000|                         0|
+----------+--------------------+----------+-----------------------------+--------------------------+

In [14]:
df_rasio_sql = spark.sql("""
    SELECT kategori,
           SUM(pendapatan) AS total_omset,
           SUM(unit_terjual) AS total_unit_terjual,
           ROUND(SUM(pendapatan) / SUM(unit_terjual), 2) AS rasio_rupiah_per_unit
    FROM view_transaksi
    GROUP BY kategori
    ORDER BY rasio_rupiah_per_unit DESC
""")

print("=== EKSPLORASI 3: ANALISIS KATEGORI PRODUK BERMARGIN TERTINGGI (SPARK SQL) ===")
df_rasio_sql.show(truncate=False)

=== EKSPLORASI 3: ANALISIS KATEGORI PRODUK BERMARGIN TERTINGGI (SPARK SQL) ===
+----------------------+-----------+------------------+---------------------+
|kategori              |total_omset|total_unit_terjual|rasio_rupiah_per_unit|
+----------------------+-----------+------------------+---------------------+
|Kesehatan & Kecantikan|41625000   |491               |84775.97             |
|Fashion               |37325000   |442               |84445.7              |
|Rumah Tangga          |37375000   |444               |84177.93             |
|Makanan & Minuman     |40975000   |525               |78047.62             |
|Elektronik            |38925000   |513               |75877.19             |
+----------------------+-----------+------------------+---------------------+



In [15]:
df_eksplorasi_4 = spark.sql("""
    SELECT kota,
           COUNT(order_id) AS total_nota_masuk,
           ROUND((COUNT(order_id) / 500.0) * 100, 2) AS persentase_kontribusi_nota
    FROM view_transaksi
    GROUP BY kota
    ORDER BY total_nota_masuk DESC
""")

print("=== EKSPLORASI 4: PERSENTASE KONTRIBUSI AKTIVITAS NOTA TRANSAKSI PER WILAYAH ===")
df_eksplorasi_4.show()

=== EKSPLORASI 4: PERSENTASE KONTRIBUSI AKTIVITAS NOTA TRANSAKSI PER WILAYAH ===
+----------+----------------+--------------------------+
|      kota|total_nota_masuk|persentase_kontribusi_nota|
+----------+----------------+--------------------------+
| Purworejo|             116|                     23.20|
|Yogyakarta|             110|                     22.00|
|      Solo|              95|                     19.00|
|  Semarang|              93|                     18.60|
|  Magelang|              86|                     17.20|
+----------+----------------+--------------------------+



In [16]:
df_eksplorasi_4 = spark.sql("""
    SELECT kota,
           COUNT(order_id) AS total_nota_masuk,
           ROUND((COUNT(order_id) / 500.0) * 100, 2) AS persentase_kontribusi_nota
    FROM view_transaksi
    GROUP BY kota
    ORDER BY total_nota_masuk DESC
""")

print("=== EKSPLORASI 4: PERSENTASE KONTRIBUSI AKTIVITAS NOTA TRANSAKSI PER WILAYAH ===")
df_eksplorasi_4.show()

=== EKSPLORASI 4: PERSENTASE KONTRIBUSI AKTIVITAS NOTA TRANSAKSI PER WILAYAH ===
+----------+----------------+--------------------------+
|      kota|total_nota_masuk|persentase_kontribusi_nota|
+----------+----------------+--------------------------+
| Purworejo|             116|                     23.20|
|Yogyakarta|             110|                     22.00|
|      Solo|              95|                     19.00|
|  Semarang|              93|                     18.60|
|  Magelang|              86|                     17.20|
+----------+----------------+--------------------------+



In [17]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
